# Template for coursework - Part 1 Classification

In [ ]:
# import of libraries
import numpy

## Team identification

* seminar day and time: Monday 18:00 - 19:30
* team number:
* names of team members: Anna Kopecny, Alisha Utegenova, Assylbek Omarov, Adil Zhumagaliyev

# Introduction

1.	Describe the business value of addressing this problem with machine learning. 

Answer: It costs significantly more to acquire a new customer than to retain an existing one. Predicting churn allows the business to proactively offer targeted promotions to at-risk customers, maximizing revenue.

2.	Provide the link to the source of the data. 
Link: https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset 

## Customization

1.	What is the *target attribute*: **Churn**
2.	What is the *instance of interest*   The instance can be identified, for example, by an id.
3.	Choose an *attribute of interest* 
5.	Show *Cost matrix* consisting of a cost of a false positive, false negative, true positive and true negative. Note that cost of true positive and true negative is recommended to be zero.

**Ideas:**
Target Attribute: Churn (usually Yes/No).

Instance of Interest: Select one specific CustomerID to focus on for your local explanations later.

Attribute of Interest: Choose a highly actionable feature, like Contract type (e.g., Month-to-month) or MonthlyCharges.

Cost Matrix: This is crucial for business value. Set True Positives (TP) and True Negatives (TN) to $0.

False Negative (FN): High cost. You predicted they wouldn't churn, but they did. The cost is the lost customer lifetime value (e.g., $500).

False Positive (FP): Lower cost. You predicted they would churn, so you gave them a retention discount, but they weren't actually going to leave. The cost is the price of the wasted promotion (e.g., $50).

In [9]:
# pip install -r requirements.txt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Data Exploration

In [14]:
#!pip install -r requirements.txt

df = pd.read_csv('C:\\Users\\I770913\\Documents\\Customer-Churn\\data\\customer_churn_dataset-training-master.csv')
df.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


1. Describe meaning of individual attributes
2.	Show a histogram (or a table with value frequencies) for the target variable and for selected other variables
3.	Show a scatterplot (correlation plot) showing the relation between selected predictors and the target variable
4.	Interpret the results


## 1. Attribute Meanings

| Column | Description |
|---|---|
| **CustomerID** | Unique identifier for each customer |
| **Age** | Customer's age in years |
| **Gender** | Customer's gender (Male / Female) |
| **Tenure** | Number of months the customer has been with the company |
| **Usage Frequency** | How frequently the customer uses the service (times per month) |
| **Support Calls** | Number of calls made to customer support |
| **Payment Delay** | Number of days the customer delayed payment |
| **Subscription Type** | Tier of subscription: Basic, Standard, or Premium |
| **Contract Length** | Duration of the contract: Monthly, Quarterly, or Annual |
| **Total Spend** | Total amount of money spent by the customer |
| **Last Interaction** | Number of days since the customer's last interaction with the company |
| **Churn** | **Target variable** — 1 if the customer churned, 0 if retained |

## 2. Basic Dataset Information

We use `.info()` and `.describe()` to understand the data types, missing values, and summary statistics.

In [ ]:
# Dataset shape
print(f"Training set shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

# Data types and non-null counts
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

## 3. Visualizations

### 3a. Distribution of the Target Variable (Churn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of churn counts
churn_counts = df["Churn"].value_counts().sort_index()
colors = ["#2ecc71", "#e74c3c"]
axes[0].bar(["Retained (0)", "Churned (1)"], churn_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Churn Class Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Number of Customers")
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 1000, f"{v:,}", ha="center", fontweight="bold")

# Pie chart showing class proportions
axes[1].pie(churn_counts.values, labels=["Retained (0)", "Churned (1)"],
            autopct="%1.1f%%", colors=colors, startangle=90, explode=(0, 0.05))
axes[1].set_title("Churn Class Proportions", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Class balance — Retained: {churn_counts.iloc[0]:,}  |  Churned: {churn_counts.iloc[1]:,}")
print(f"Churn rate: {churn_counts.iloc[1] / len(df) * 100:.1f}%")

### 3b. Distributions of Key Numeric Variables

In [ ]:
numeric_cols = ["Age", "Tenure", "Usage Frequency", "Support Calls",
                "Payment Delay", "Total Spend", "Last Interaction"]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(data=df, x=col, hue="Churn", kde=True, ax=axes[i],
                 palette={0: "#2ecc71", 1: "#e74c3c"}, alpha=0.6)
    axes[i].set_title(f"Distribution of {col}", fontsize=12, fontweight="bold")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")

# Hide the unused subplot
axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

### 3c. Frequency Tables for Categorical Variables

In [ ]:
cat_cols = ["Gender", "Subscription Type", "Contract Length"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df["Churn"], normalize="index") * 100
    ct.plot(kind="bar", stacked=True, ax=axes[i],
            color=["#2ecc71", "#e74c3c"], edgecolor="black")
    axes[i].set_title(f"Churn Rate by {col}", fontsize=13, fontweight="bold")
    axes[i].set_ylabel("Percentage (%)")
    axes[i].set_xlabel(col)
    axes[i].legend(["Retained (0)", "Churned (1)"], loc="upper right")
    axes[i].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

### 3d. Scatterplot — Tenure vs. Total Spend (colored by Churn)

In [ ]:
# Use a sample for readability (full dataset is 440K+ rows)
sample_df = df.sample(n=5000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Tenure vs Total Spend
sns.scatterplot(data=sample_df, x="Tenure", y="Total Spend", hue="Churn",
                palette={0: "#2ecc71", 1: "#e74c3c"}, alpha=0.5, ax=axes[0])
axes[0].set_title("Tenure vs. Total Spend (by Churn)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Tenure (months)")
axes[0].set_ylabel("Total Spend")

# Usage Frequency vs Support Calls
sns.scatterplot(data=sample_df, x="Usage Frequency", y="Support Calls", hue="Churn",
                palette={0: "#2ecc71", 1: "#e74c3c"}, alpha=0.5, ax=axes[1])
axes[1].set_title("Usage Frequency vs. Support Calls (by Churn)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Usage Frequency")
axes[1].set_ylabel("Support Calls")

plt.tight_layout()
plt.show()

### 3e. Correlation Matrix Heatmap

In [ ]:
# Correlation matrix for all numeric columns (including the target)
numeric_df = df[numeric_cols + ["Churn"]]
corr_matrix = numeric_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # mask the upper triangle
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdYlGn_r",
            mask=mask, vmin=-1, vmax=1, linewidths=0.5,
            cbar_kws={"label": "Pearson Correlation"})
plt.title("Correlation Matrix — Numeric Features + Churn", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 3f. Boxplots — Key Features by Churn Status

In [ ]:
box_cols = ["Tenure", "Total Spend", "Support Calls", "Payment Delay"]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for i, col in enumerate(box_cols):
    sns.boxplot(data=df, x="Churn", y=col, ax=axes[i],
                palette={0: "#2ecc71", 1: "#e74c3c"})
    axes[i].set_title(f"{col} by Churn", fontsize=12, fontweight="bold")
    axes[i].set_xticklabels(["Retained (0)", "Churned (1)"])
    axes[i].set_xlabel("")

plt.tight_layout()
plt.show()

## 4. Interpretation of Exploratory Analysis

**Key observations from the data exploration:**

1. **Class imbalance:** The target variable `Churn` shows a roughly 50/50 split in this dataset (the exact ratio is visible in the bar/pie chart above). This is relatively balanced, which simplifies modeling — no resampling may be strictly necessary, though it should still be monitored.

2. **Tenure and Total Spend:** Customers who churn tend to have **shorter tenures** and **lower total spend**. This makes intuitive sense — newer customers who haven't invested much in the service are more likely to leave. The scatterplot shows churned customers clustering toward the lower-left (low tenure, low spend).

3. **Support Calls:** Churned customers make **significantly more support calls** on average. Frequent support interactions may signal dissatisfaction with the service, making this one of the strongest predictors of churn.

4. **Payment Delay:** Customers who churn tend to have **longer payment delays**. Late payments may indicate the customer is already disengaging from the service.

5. **Usage Frequency:** Lower usage frequency correlates with higher churn — customers who use the service less have less reason to stay.

6. **Contract Length:** Monthly contracts are expected to exhibit higher churn rates compared to quarterly or annual contracts, as there is less commitment locking the customer in.

7. **Correlation matrix:** Most features show low pairwise correlation with each other, suggesting they provide **independent signals** for churn prediction. The strongest correlates with `Churn` are likely `Support Calls`, `Payment Delay`, and `Tenure` (visible from the heatmap).

> **Summary:** The typical churning customer profile is one with a **short tenure, high support call volume, frequent payment delays, low usage, and a monthly contract**. These patterns will guide our feature engineering and model selection in the next steps.

# Data preprocessing

## Preprocessing for supervised machine learning 

### Derive binary target attribute (if not already binary)

### Train test split 

### Feature engineering 

Do at least one additional data preprocessing, such as

-	Remove missing values
-	If the classes are imbalanced, you may upsample (or downsample) the training dataset.
-	Normalize values or use a standard scale 
-	Remove rows based on subsetting
-	Derive new columns
-	Perform feature selection (remove some attributes)


Make sure that your preprocessing operation does not use information from the test set. It is therefore recommended to “fit” preprocessing on the training set and then apply it on the test set.

## Applying preprocessing on test data

## Modeling

* Train the model on training data, and evaluate the model on test data. 
* Try at least two machine learning classification algorithms. It is recommended to try Decision Trees and Forests. 


### Classifier 1 (decision tree)

* Try various combinations of metaparameters (such as tree depth for decision tree) and record the impact on predictive performance. You can use grid search cross-validation for this.
* Once you determine the best values, you can refit the model with the best parameter value on the entire training data.

### Classifier 2 (random forest)

* Try various combinations of metaparameters (such as  number of trees in a forest) and record the impact on predictive performance. You can use grid search cross-validation for this.
* Once you determine the best values, you can refit the model with the best parameter value on the entire training data.


### Classifier 3 (baseline)

* Fit a baseline model, e.g., a model that predicts the most frequent class in the training data

# Evaluation

### Classifier 1 (decision tree)

* Compute accuracy and F1 score on test data (you can include also other measures)
* Show confusion matrix
* Multiply the predefined costs with the confusion matrix to get the overall cost of the model

### Classifier 2 (random forest)

* Compute accuracy and F1 score on test data (you can include also other measures)
* Show confusion matrix
* Multiply the predefined costs with the confusion matrix to get the overall cost of the model

### Classifier 3 (baseline)

* Compute accuracy and F1 score on test data (you can include also other measures)
* Show confusion matrix
* Multiply the predefined costs with the confusion matrix to get the overall cost of the model

### Summary

* Which metric is most suitable for use for the current problem (accuracy, F-measure)?
* Compare the performance metrics for all types of models (e.g,. decision tree and forest). Which model is the best one?
* Combine (multiply) the predefined costs matrix with the values in the confusion matrix for each model. Which model is the best one? 


# Explanation

## Global explanation

### Classifier 1 - decision tree

* Visualize the decision tree
* Looking at the tree, list the most important attributes

### Classifier 2 - random forest

* Show the feature importance of variables in the forest

## Local explanation

* Show the *instance of interest* - a row in the dataframe
* Use both models to classify the chosen instance
* Do both models assign the same class?
* What is the confidence (probability) of the prediction?
* If you change the value of the attribute of interest in the instance of interest, how does the classification of the instance change? 

# Conclusion
Summarize the results, answering questions such as:

1.	Which machine learning result has the highest value and is most interesting? 
2.	What setting provided the best result? 
3.	Which attributes are the most important?


# Optional parts

## Evaluation  - cost based

### Ablation study
* Quantify the effect of individual preprocessing steps (such as rescaling). How would the performance change if you have not performed this step (optional).

###  Optimization of threshold (optional)

* If you would change the probability (score) threshold for classification, would you obtain better results in terms of total costs? For which threshold? 

## Explanation

*	Apply ICE/IME/SHAPLEY/Anchors to explain the classification of the instance

# Final checklist

-	Are all preprocessing steps justified?
-	Did you try different metaparameter values where appropriate?
-	Are the results replicable? If you have the same data, does the report describe all steps in sufficient detail to obtain the same results as reported by the authors?
-	Were proper evaluation metrics selected? Are the results correctly interpreted?
-	Are all important steps explained and justified?
-	What is the quality of writing? Is the language clear and concise?


# Submission

* This .ipynb file with your code + its html version after the code was run (File-Save and export notebook as - html)
* Source data or a link to source data or source data being loaded from a url in the notebook
* Data files after preprorcessing (train.csv and test.csv)